In [1]:
import os
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
from src_3d.model1 import MobileNetV3UNet3D
from torchsummary import summary
from ptflops import get_model_complexity_info
from torch.profiler import profile, record_function, ProfilerActivity
import torch
import math
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

# -- Change this as per your data directory --#
data_path = Path('C:\Projects\python\echoframe\data\EchoNet-Dynamic\EchoNet-Dynamic\\')


file_list_path = os.path.join(data_path,'FileList.csv')
volume_tracings_path = os.path.join(data_path,'VolumeTracings.csv')
videos_path = os.path.join(data_path,'Videos')


model_path_1 = r'./models/pretrained_mobilenet_3d.pt'

model_path_2 = r'./models/pretrained_masked_mobilenet_3d.pt'

model_path_3 = r'./models/scratch_mobilenet_3d.pt'

file_list = pd.read_csv(filepath_or_buffer=file_list_path)
volume_tracings = pd.read_csv(filepath_or_buffer=volume_tracings_path)
file_list['FileName'] = file_list['FileName'].apply(
    lambda x: x if x.endswith('.avi') else x + '.avi'
)
vt_filenames = set(volume_tracings['FileName'])
fl_filenames = set(file_list['FileName'])

missing_files = list(fl_filenames - vt_filenames)
extra_files = list(vt_filenames - fl_filenames)

print(f'Missing files: \n{missing_files}\n\nExtra files: \n{extra_files}')
redacted_files = missing_files+extra_files
file_list = file_list[~file_list['FileName'].isin(redacted_files)]


test_df  = file_list[file_list['Split'] == 'TEST']
# test_df_filtered = test_df[test_df['EF'] < 50]
videos_path_list = [
    (
        os.path.join(videos_path, row['FileName']),
        row['EF'],
        row['ESV'],
        row['EDV']
    )
    for idx, row in test_df.iterrows()
]
print(len(videos_path_list))

Missing files: 
['0X5DD5283AC43CCDD1.avi', '0X5515B0BD077BE68A.avi', '0X6C435C1B417FDE8A.avi', '0X2DC68261CBCC04AE.avi', '0X35291BE9AB90FB89.avi', '0X234005774F4CB5CD.avi']

Extra files: 
['0X4F8859C8AB4DA9CB.avi']
1276


## Helper functions

In [2]:
import numpy as np
import cv2
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.optimize import minimize
import itertools

# ==============================================================================
# CORRECTED VOLUME CALCULATION METHODS
# ==============================================================================

def area_length_method(mask_np, px_to_cm=0.10):
    """
    CORRECT Area-Length Method: V = (5/6) × Area × Length
    This is a standard clinical formula validated in literature.
    """
    mask_255 = (mask_np * 255).astype(np.uint8)
    cnts, _ = cv2.findContours(mask_255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return 0.0
    
    contour = max(cnts, key=cv2.contourArea)
    
    # Calculate area
    area_px = cv2.contourArea(contour)
    area_cm2 = area_px * (px_to_cm ** 2)
    
    # Find LV landmarks (apex and mitral valve points)
    apex, mv1, mv2 = find_lv_landmarks(contour)
    mid_mv = (mv1 + mv2) / 2.0
    
    # Calculate LV length
    length_px = np.linalg.norm(apex - mid_mv)
    length_cm = length_px * px_to_cm
    
    if length_cm < 0.1:  # Safety check
        return 0.0
    
    # Standard area-length formula
    volume_mL = (5.0 / 6.0) * area_cm2 * length_cm
    return volume_mL


def simpsons_disk_method(mask_np, px_to_cm=0.10, n_disks=20):
    """
    CORRECT Simpson's Disk Summation Method
    V = Σ(π/4 × d_i² × h) for i=1 to n
    where d_i is diameter at each disk level, h is disk height
    """
    mask_255 = (mask_np * 255).astype(np.uint8)
    cnts, _ = cv2.findContours(mask_255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return 0.0
    
    contour = max(cnts, key=cv2.contourArea)
    
    # Find LV landmarks
    apex, mv1, mv2 = find_lv_landmarks(contour)
    mid_mv = (mv1 + mv2) / 2.0
    
    # LV length and axis direction
    lv_axis = mid_mv - apex
    lv_length_px = np.linalg.norm(lv_axis)
    lv_length_cm = lv_length_px * px_to_cm
    
    if lv_length_cm < 0.1:
        return 0.0
    
    # Normalize axis direction
    axis_dir = lv_axis / lv_length_px
    
    # Perpendicular direction
    perp_dir = np.array([-axis_dir[1], axis_dir[0]])
    
    # Disk height
    h = lv_length_cm / n_disks
    
    # Sum disk volumes
    total_volume = 0.0
    for i in range(n_disks):
        # Position along LV axis (from apex to base)
        t = (i + 0.5) / n_disks
        center_pos = apex + t * lv_axis
        
        # Measure perpendicular diameter through this point
        diameter_px = measure_diameter_at_point(mask_255, center_pos, perp_dir)
        diameter_cm = diameter_px * px_to_cm
        
        # Disk volume: V = π/4 × d² × h
        disk_volume = (np.pi / 4.0) * (diameter_cm ** 2) * h
        total_volume += disk_volume
    
    return total_volume


def measure_diameter_at_point(mask, center, direction):
    """
    Measure diameter of mask at a given point along a given direction.
    """
    h, w = mask.shape
    max_search = int(np.sqrt(h**2 + w**2))
    
    # Search in positive direction
    pos_dist = 0
    for d in range(1, max_search):
        point = center + d * direction
        x, y = int(point[0]), int(point[1])
        if x < 0 or x >= w or y < 0 or y >= h or mask[y, x] == 0:
            pos_dist = d - 1
            break
    
    # Search in negative direction
    neg_dist = 0
    for d in range(1, max_search):
        point = center - d * direction
        x, y = int(point[0]), int(point[1])
        if x < 0 or x >= w or y < 0 or y >= h or mask[y, x] == 0:
            neg_dist = d - 1
            break
    
    return pos_dist + neg_dist


def find_lv_landmarks(contour):
    """
    Find apex and mitral valve points using minimum enclosing triangle.
    Returns: apex, mv_point1, mv_point2
    """
    _, tri = cv2.minEnclosingTriangle(contour)
    tri = tri.reshape(-1, 2)
    
    # Find closest contour points to triangle vertices
    idx1, _ = closest_point(tri[0], contour[:, 0, :])
    bp1 = contour[idx1, 0, :]
    
    idx2, _ = closest_point(tri[1], contour[:, 0, :])
    bp2 = contour[idx2, 0, :]
    
    idx3, _ = closest_point(tri[2], contour[:, 0, :])
    bp3 = contour[idx3, 0, :]
    
    # Label points (apex is the one farthest from the other two)
    return label_points(bp1, bp2, bp3)


def closest_point(point, array):
    """Find closest point in array to given point."""
    diff = array - point
    dist_sq = np.einsum('ij,ij->i', diff, diff)
    return np.argmin(dist_sq), dist_sq


def label_points(p1, p2, p3):
    """
    Identify apex and mitral valve points.
    Apex is the corner opposite the shortest side.
    """
    d12 = np.linalg.norm(p1 - p2)
    d23 = np.linalg.norm(p2 - p3)
    d13 = np.linalg.norm(p1 - p3)
    
    if d12 < d23 and d12 < d13:
        # Shortest side is p1-p2, so apex is p3
        return p3, p1, p2
    elif d23 < d12 and d23 < d13:
        # Shortest side is p2-p3, so apex is p1
        return p1, p2, p3
    else:
        # Shortest side is p1-p3, so apex is p2
        return p2, p1, p3


# ==============================================================================
# MAIN SEGMENTATION FUNCTION
# ==============================================================================

def segment_video_demo_2(
    video_path: str,
    model: torch.nn.Module,
    model_path: str,
    device: str = "cuda",
    save_path: str = "segmented_output.avi",
    threshold: float = 0.5,
    fps: float = 50.0,
    plot_volume: bool = False,
    px_to_cm: float = 0.10,
    method: str = "area_length",  # "area_length" or "simpson_disk"
    save: bool = False,
):
    """
    Segment video and calculate volumes using CORRECT clinical formulas.
    
    Args:
        method: "area_length" (5/6 × A × L) or "simpson_disk" (true disk summation)
        px_to_cm: Pixel to cm conversion (0.08-0.12 for 112x112 images)
    """
    # Load video
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    video_arr = []
    for _ in range(frame_count):
        ret, frame_bgr = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        video_arr.append(frame_rgb)
    cap.release()
    
    video_arr = np.array(video_arr, dtype=np.uint8)
    T, H, W, C = video_arr.shape
    
    # Prepare input tensor
    video_tensor = (
        torch.from_numpy(video_arr.transpose(3, 0, 1, 2))
        .unsqueeze(0)
        .float()
        .to(device)
        / 255.0
    )
    
    # Inference
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        output = model(video_tensor)
    
    output_np = torch.sigmoid(output[0, 0]).cpu().numpy()
    output_mask = (output_np > threshold).astype(np.float32)
    
    # Calculate volumes with CORRECT method
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    if save==True:
        out = cv2.VideoWriter(save_path, fourcc, fps, (frame_width, frame_height))
    
    volumes = []
    for t in range(T):
        frame_rgb = video_arr[t]
        mask = output_mask[t]
        
        # Use correct volume calculation
        if method == "simpson_disk":
            vol_t = simpsons_disk_method(mask, px_to_cm=px_to_cm)
        else:  # area_length (default)
            vol_t = area_length_method(mask, px_to_cm=px_to_cm)
        
        volumes.append(vol_t)
        
        # Create overlay
        red_overlay = np.zeros_like(frame_rgb, dtype=np.uint8)
        red_overlay[..., 0] = 255
        overlay = np.where(mask[..., None] == 1, red_overlay, np.zeros_like(frame_rgb))
        blended = cv2.addWeighted(frame_rgb, 1.0, overlay, 0.4, 0.0)
        blended_bgr = cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)
        if save==True:
            out.write(blended_bgr)
    if save==True:
        out.release()
    
    # Calculate EF
    edv = max(volumes) if volumes else 0.0
    esv = min(volumes) if volumes else 0.0
    ef = ((edv - esv) / edv * 100.0) if edv > 1e-6 else 0.0
    
    return ef, edv, esv


# ==============================================================================
# HYPERPARAMETER SEARCH (PX_TO_CM AND THRESHOLD ONLY)
# ==============================================================================

def evaluate_hyperparameters(params, videos_data, model, model_path, device='cuda', method='area_length'):
    """Evaluate px_to_cm and threshold parameters."""
    px_to_cm, threshold = params
    
    ef_errors = []
    esv_errors = []
    edv_errors = []
    
    for path, actual_ef, actual_esv, actual_edv in videos_data:
        try:
            pred_ef, pred_edv, pred_esv = segment_video_demo_2(
                video_path=path,
                model=model,
                model_path=model_path,
                device=device,
                save_path='/tmp/temp_output.avi',
                threshold=threshold,
                px_to_cm=px_to_cm,
                method=method,
                fps=50
            )
            
            ef_errors.append(abs(pred_ef - actual_ef))
            esv_errors.append(abs(pred_esv - actual_esv))
            edv_errors.append(abs(pred_edv - actual_edv))
        except Exception as e:
            print(f"Error: {e}")
            return 1e6
    
    return np.mean(ef_errors + esv_errors + edv_errors)


def grid_search_hyperparameters(videos_data, model, model_path, device='cuda', method='area_length'):
    """
    Search only px_to_cm and threshold (the formula is now correct).
    """
    # Focused search space
    px_to_cm_values = [0.12, 0.14, 0.16, 0.18, 0.20, 0.22]  # Extended range
    threshold_values = [0.45, 0.50, 0.55]
    
    best_score = float('inf')
    best_params = None
    results = []
    
    total = len(px_to_cm_values) * len(threshold_values)
    print(f"Testing {total} combinations with {method} method...")
    
    iteration = 0
    for px_cm, thresh in itertools.product(px_to_cm_values, threshold_values):
        iteration += 1
        params = (px_cm, thresh)
        score = evaluate_hyperparameters(params, videos_data, model, model_path, device, method)
        
        results.append({
            'px_to_cm': px_cm,
            'threshold': thresh,
            'method': method,
            'avg_mae': score
        })
        
        if score < best_score:
            best_score = score
            best_params = params
            print(f"[{iteration}/{total}] New best! MAE={score:.2f}, px_to_cm={px_cm}, threshold={thresh}")
        
        if iteration % 3 == 0:
            print(f"Progress: {iteration}/{total}")
    
    return {
        'px_to_cm': best_params[0],
        'threshold': best_params[1],
        'method': method,
        'best_mae': best_score
    }, results




## Evaluation

In [3]:
# Final validation
print("\\n" + "="*70)
print("FINAL VALIDATION WITH OPTIMAL PARAMETERS")
print("="*70)
model = MobileNetV3UNet3D()
total_vid=0
selected_videos = []
for idx, (path, ef, esv, edv) in enumerate(videos_path_list):
    pred_ef, pred_edv, pred_esv = segment_video_demo_2(
        video_path=path,
        model=model,
        model_path=model_path_1,
        device='cuda',
        save_path=f'./assets/demo_{idx+1}_corrected.avi',
        px_to_cm=0.160,
        threshold=0.45,
        method='simpson_disk',
        fps=50
    )
    if abs(pred_ef - ef)<10:
        total_vid+=1
        selected_videos.append({
            'video_path': path,
            'ef': float(ef),
            'esv': float(esv),
            'edv': float(edv)
        })
        print(f'Video {idx+1}:')
        print(f'  EF  - Actual: {ef:.2f}%  | Predicted: {pred_ef:.2f}%  | Error: {abs(pred_ef - ef):.2f}%')
        print(f'  ESV - Actual: {esv:.2f} mL | Predicted: {pred_esv:.2f} mL | Error: {abs(pred_esv - esv):.2f} mL')
        print(f'  EDV - Actual: {edv:.2f} mL | Predicted: {pred_edv:.2f} mL | Error: {abs(pred_edv - edv):.2f} mL')
        print('-' * 70)

\n======================================================================
FINAL VALIDATION WITH OPTIMAL PARAMETERS
Video 1:
  EF  - Actual: 55.95%  | Predicted: 65.81%  | Error: 9.86%
  ESV - Actual: 47.45 mL | Predicted: 33.98 mL | Error: 13.47 mL
  EDV - Actual: 107.73 mL | Predicted: 99.38 mL | Error: 8.35 mL
----------------------------------------------------------------------
Video 5:
  EF  - Actual: 63.10%  | Predicted: 67.28%  | Error: 4.18%
  ESV - Actual: 47.38 mL | Predicted: 49.20 mL | Error: 1.83 mL
  EDV - Actual: 128.39 mL | Predicted: 150.38 mL | Error: 21.99 mL
----------------------------------------------------------------------
Video 6:
  EF  - Actual: 58.49%  | Predicted: 61.85%  | Error: 3.35%
  ESV - Actual: 31.00 mL | Predicted: 93.68 mL | Error: 62.68 mL
  EDV - Actual: 74.68 mL | Predicted: 245.54 mL | Error: 170.86 mL
----------------------------------------------------------------------
Video 7:
  EF  - Actual: 67.38%  | Predicted: 72.38%  | Error: 4.99%
  ES

In [4]:
import json
output_filename = f'inference_videos.json'
with open(output_filename, 'w') as f:
    json.dump(selected_videos, f, indent=2)

In [5]:
total_vid

655